In [13]:
# get groundtruth label for all images

import glob
import os
import pickle

output_dir = '/data/haozhen/uouo/mosaic_output'
gt_output_dir = os.path.join(output_dir, 'model-cascade-small-vlm-shuffle-rag')

image_path = os.path.join(gt_output_dir, 'Mosaic-Image')
img_id_dict_path = '/data/haozhen/uouo/UOUO/mosaic/preprocess/category_lookup.pkl'

def generate_gt_mosaic_labels(image_path, output_dir, id_dict_path='../category_lookup.pkl'):
    files = glob.glob(image_path + '/**/*', recursive=True)
    img_paths = []
    for file in files:
        if os.path.isfile(file):
            img_paths.append(file)

    with open(id_dict_path, 'rb') as file:
        img_id_dict = pickle.load(file)

    gt_label_dict = dict()

    for img_path in img_paths:
        filename = img_path.split('/')[-1]
        four_id = filename.split('.')[0].split('_')[1:]
        labels = [img_id_dict[four_id[0]], 
                    img_id_dict[four_id[1]],
                    img_id_dict[four_id[2]],
                    img_id_dict[four_id[3]]]
        gt_label_dict[filename] = {'labels': labels, 'img_path': img_path}
        print(gt_label_dict[filename])

    with open(os.path.join(gt_output_dir,'gt_mosaic_img_labels.pkl'), 'wb') as file:
        pickle.dump(gt_label_dict, file)

    print(f'groundtruth label file is successfully generated.')
    
generate_gt_mosaic_labels(image_path, gt_output_dir, img_id_dict_path)

    




{'labels': ['Grape harvester', 'PTO driven chipper shredder', 'Mixer wagon', 'Spike driving machine'], 'img_path': '/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag/Mosaic-Image/19/mo_079110036_121920018_035220000_094210032.png'}
{'labels': ['Streak Camera', 'Air Table for Paper Handling', 'Bead Wire Feeder', 'Surge Tester'], 'img_path': '/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag/Mosaic-Image/306/mo_097420039_001910015_012100030_175840001.png'}
{'labels': ['Legger press', 'Surface polishing machine', 'Charpy impact tester', 'Book Backing Machine'], 'img_path': '/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag/Mosaic-Image/73/mo_101180026_098640020_030380037_009850030.png'}
{'labels': ["Vintner's Scoop", 'Vichy shower', 'Adjustable Bed', 'Burial vault'], 'img_path': '/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag/Mosaic-Image/156/mo_194950023_194540042_001070008_021870004.png'}
{'labels': ['Wire gui

In [61]:
# get all categories 
# with open(img_id_dict_path, 'rb') as file:
#     id_dict = pickle.load(file)

# categories = set()
# for cat in id_dict.values():
#     categories.add(cat)

# categories = list(categories)  # 406 categories
# print(categories)
# with open(os.path.join(output_dir,'categories.pkl'), 'wb') as file:
#         pickle.dump(categories, file)

In [15]:
with open(os.path.join(output_dir, 'category_purposes_cleaned.pkl'), 'rb') as file:
    purposes_dict = pickle.load(file)

    with open(os.path.join(gt_output_dir, 'gt_mosaic_img_labels.pkl'), 'rb') as file2:
        gt_label_dict = pickle.load(file2)

        for cat in gt_label_dict.keys():
            labels = gt_label_dict[cat]["labels"]
            gt_label_dict[cat]["purposes"] = [purposes_dict[l] for l in labels]
            
        with open(os.path.join(gt_output_dir, 'gt_mosaic_img_labels_purpose.pkl'), 'wb') as file3:
            pickle.dump(gt_label_dict, file3)




In [ ]:
# generate question list
purpose_template_small_vlm = """Identify the location of the given object for {purpose}.
The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right'. 
Only give response as one of the possible answers."""

object_template_small_vlm = """Identify the location of the given object in this 2x2 mosaic image. 
The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right'. 
Only give response as one of the possible answers.\n
object name: {cat}\nLocation: """

purpose_template_gpt_desc = """Identify the location of the given object for {purpose}.
The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right'. 
There is only one answer possible.
Please include your reasoning steps, then answer your choice in this format: ANSWER: <answer>."""

object_template_gpt_desc = """Identify the location of {cat} in this 2x2 mosaic image. 
The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right'. 
There is only one answer possible.
Please include your reasoning steps, then answer your choice in this format: ANSWER: <answer>."""

purpose_in_option_small_vlm ="""What is the purpose of the object at the {position} of the image? 
The possible options are A.{option_1}, B.{option_2}, C.{option_3}, D.{option_4}.
There is only one option possible.
Respond with only the letter corresponding to the correct option: A, B, C, or D.
"""

purpose_in_option_gpt_desc ="""What is the purpose of the object at the {position} of the image? 
The possible options are A.{option_1}, B.{option_2}, C.{option_3}, D.{option_4}.
There is only one option possible.
Please include your reasoning steps, then answer your choice corresponding to the correct option: A, B, C, or D in this format: ANSWER: <answer>.
"""


In [ ]:
input: {labels, img_path}, {img_path:det_id}, {label: 5 purposes}
output: {question, img_path}

In [ ]:
import random

# {label: [5 purposes]}
purposes_dict_path = '/data/haozhen/uouo/mosaic_output/category_purposes_cleaned.pkl'
with open(purposes_dict_path, 'rb') as file:
    purposes_dict = pickle.load(file)

# {labels, img_path}
output_dir = '/data/haozhen/uouo/mosaic_output'
gt_output_dir = os.path.join(output_dir, 'model-cascade-small-vlm-shuffle-rag')

with open(os.path.join(gt_output_dir,'gt_mosaic_img_labels.pkl'), 'rb') as file:
    gt_mosaic_labels = pickle.load(file)

# {img_path:det_id}
with open(os.path.join(gt_output_dir, "dt_img_id.pkl"), "rb") as file:
    dt_id_dict = pickle.load(file)

uouo_gpt_queries = []
pos_dict = {0: "top left", 1: 'top right', 2: 'bottom left', 3: 'bottom right'} # id: position
for mosaic_filename, mosaic_item in gt_mosaic_labels.items():
    labels = mosaic_item['labels']
    img_path = mosaic_item['img_path']
    dt_id = dt_id_dict[mosaic_filename]
    dt_label = labels[dt_id]
    dt_purposes = purposes_dict[dt_label]
    chosen_purpose_1 = random.choice(dt_purposes)

    # get options for purpose-in-option queries
    chosen_purpose_2 = random.choice(dt_purposes)
    opt_num_dict = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    all_id = [0, 1, 2, 3]
    other_three_id = [x for x in all_id if x != dt_id]
    other_three_purpose = [random.choice(purposes_dict[labels[id]]) for id in other_three_id]
    purpose_option =other_three_purpose + [chosen_purpose_2]
    random.shuffle(purpose_option)
    gt_opt_num_idx = purpose_option.index(chosen_purpose_2)
    gt_opt_num = opt_num_dict[gt_opt_num_idx]


    purpose_template_small_vlm_prompt = purpose_template_small_vlm.format(purpose=chosen_purpose_1)
    object_template_small_vlm_prompt = object_template_small_vlm.format(cat=dt_label) 
    purpose_in_option_small_vlm_prompt = purpose_in_option_small_vlm.format(option_1=purpose_option[0],
                                                                          option_2=purpose_option[1],
                                                                          option_3=purpose_option[2],
                                                                          option_4=purpose_option[3],
                                                                          position=pos_dict[dt_id])

    query = {'img_path': img_path, 
             'purpose_prompt': purpose_template_small_vlm_prompt, # given purpose, ask where
             'object_prompt': object_template_small_vlm_prompt, # given object name, ask where
             'purpose_in_option_prompt': purpose_in_option_small_vlm_prompt, # given position, ask which purpose
             'dt_id': dt_id,   # which object should be query, pos_id from 1 2 3 4 
             'dt_label': dt_label, # which object should be query, object name
             'dt_purpose': chosen_purpose_1,  # for purpose_prompt groundtruth purpose
             'gt_position': pos_dict[dt_id],  # the determined query object location, top left ...
             'purpose_in_opt_gt_ans_ABCD' : gt_opt_num, # gt answer choice ABCD for purpose_in_option_prompt
             'purpose_in_opt_gt_ans': chosen_purpose_2, # gt answer purpose for purpose_in_option_prompt
             'purpose_in_opt_options': purpose_option} #  all optiosn for purpose_in_option_prompt
    
    print(query)
    uouo_gpt_queries.append(query)


with open(os.path.join(gt_output_dir, 'uouo_test_queries.pkl'), 'wb') as file:
            pickle.dump(uouo_gpt_queries, file)







{'mo_169860015_016070019_160990008_003730011.png': 1, 'mo_111240018_172840012_158360007_099830007.png': 1, 'mo_058610023_031980005_186900035_165120008.png': 0, 'mo_033280017_096640030_158370006_100990039.png': 3, 'mo_127770015_099310006_069910038_036340019.png': 3, 'mo_195670019_068090015_195670037_008160013.png': 1, 'mo_098640016_011940021_004610030_067010012.png': 0, 'mo_182770030_013880007_096640022_167770024.png': 1, 'mo_183150017_019610016_021870008_184710031.png': 3, 'mo_048620022_165120018_008740013_011940027.png': 2, 'mo_158360012_015320025_111240016_101810026.png': 3, 'mo_012100036_108680031_031980005_098050011.png': 1, 'mo_016030026_084230007_032740010_015320015.png': 3, 'mo_153240020_019610033_040740017_021870036.png': 1, 'mo_040150012_042600001_080180020_121920010.png': 0, 'mo_154640000_065940027_133140020_021950001.png': 0, 'mo_110040034_137150018_009850006_086860020.png': 1, 'mo_044810018_183670028_130880020_119770013.png': 2, 'mo_079110036_121920018_035220000_094210032.p

In [ ]:
import random

# {label: [5 purposes]}
purposes_dict_path = '/data/haozhen/uouo/mosaic_output/category_purposes_cleaned.pkl'
with open(purposes_dict_path, 'rb') as file:
    purposes_dict = pickle.load(file)

# {labels, img_path}
output_dir = '/data/haozhen/uouo/mosaic_output'
gt_output_dir = os.path.join(output_dir, 'model-cascade-big-vlm-shuffle-rag')

with open(os.path.join(gt_output_dir,'gt_mosaic_img_labels.pkl'), 'rb') as file:
    gt_mosaic_labels = pickle.load(file)

# {img_path:det_id}
with open(os.path.join(gt_output_dir, "dt_img_id.pkl"), "rb") as file:
    dt_id_dict = pickle.load(file)

uouo_gpt_queries = []
pos_dict = {0: "top left", 1: 'top right', 2: 'bottom left', 3: 'bottom right'} # id: position
for mosaic_filename, mosaic_item in gt_mosaic_labels.items():
    labels = mosaic_item['labels']
    img_path = mosaic_item['img_path']
    dt_id = dt_id_dict[mosaic_filename]
    dt_label = labels[dt_id]
    dt_purposes = purposes_dict[dt_label]
    chosen_purpose_1 = random.choice(dt_purposes)

    # get options for purpose-in-option queries
    chosen_purpose_2 = random.choice(dt_purposes)
    opt_num_dict = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    all_id = [0, 1, 2, 3]
    other_three_id = [x for x in all_id if x != dt_id]
    other_three_purpose = [random.choice(purposes_dict[labels[id]]) for id in other_three_id]
    purpose_option =other_three_purpose + [chosen_purpose_2]
    random.shuffle(purpose_option)
    gt_opt_num_idx = purpose_option.index(chosen_purpose_2)
    gt_opt_num = opt_num_dict[gt_opt_num_idx]


    purpose_template_gpt_desc_prompt = purpose_template_gpt_desc.format(purpose=chosen_purpose_1)
    object_template_gpt_desc_prompt = object_template_gpt_desc.format(cat=dt_label) 
    purpose_in_option_gpt_desc_prompt = purpose_in_option_gpt_desc.format(option_1=purpose_option[0],
                                                                          option_2=purpose_option[1],
                                                                          option_3=purpose_option[2],
                                                                          option_4=purpose_option[3],
                                                                          position=pos_dict[dt_id])

    query = {'img_path': img_path, 
             'purpose_prompt': purpose_template_gpt_desc_prompt, # given purpose, ask where
             'object_prompt': object_template_gpt_desc_prompt, # given object name, ask where
             'purpose_in_option_prompt': purpose_in_option_gpt_desc_prompt, # given position, ask which purpose
             'dt_id': dt_id,   # which object should be query, pos_id from 1 2 3 4 
             'dt_label': dt_label, # which object should be query, object name
             'dt_purpose': chosen_purpose_1,  # for purpose_prompt groundtruth purpose
             'gt_position': pos_dict[dt_id],  # the determined query object location, top left ...
             'purpose_in_opt_gt_ans_ABCD' : gt_opt_num, # gt answer choice ABCD for purpose_in_option_prompt
             'purpose_in_opt_gt_ans': chosen_purpose_2, # gt answer purpose for purpose_in_option_prompt
             'purpose_in_opt_options': purpose_option} #  all optiosn for purpose_in_option_prompt
    
    print(query)
    uouo_gpt_queries.append(query)


with open(os.path.join(gt_output_dir, 'uouo_gpt_queries.pkl'), 'wb') as file:
            pickle.dump(uouo_gpt_queries, file)







FileNotFoundError: [Errno 2] No such file or directory: '/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag/gt_mosaic_img_labels.pkl'